<div align="center">

<a href="https://colab.research.google.com/github/utkukose/cyber-physical-systems-NB-lecture/blob/main/exams/midterm/MID1_heated_room.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm option 1: A heated room

**Cyber-Physical Systems (BIL-435 Siber Fiziksel Sistemler)**<br>Midterm project: A model and a controller for a cyber-physical system. Starter Colab notebook

[Project page](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/exams/midterm) | [Course repository](https://github.com/utkukose/cyber-physical-systems-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: A seminar room with an electric heater of 8 kW on a winter day. Replace the values marked `# TODO` with the data of your own system. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. The room as a heat capacitor

*Lecture: [Week 2, A building as a heat capacitor](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#a-building-as-a-heat-capacitor)*

The room stores heat in its air, walls and furniture. Its thermal capacitance $C$ is given in kWh/K. Heat leaks to the
outdoor air through the thermal resistance $R$ in K/kW. The heater adds the power $P$, and people, lights and devices
add the internal gains $P_i$, both in kW. The indoor temperature $T$ in °C follows

$$C\,\frac{dT}{dt} = \frac{T_{out} - T}{R} + P + P_i .$$

With constant inputs, $T$ approaches $T_\infty = T_{out} + R\,(P + P_i)$ with the time constant $\tau = R\,C$ in hours.
A step test measures this curve. The heater runs at full power in the empty room, and the temperature is logged every
10 minutes. A least-squares fit of the curve gives $T_\infty$ and $\tau$, and from them $R$ and $C$. The root mean
square error (RMSE) of the fit shows how well the model matches the log.

In [ ]:
# TODO: the step test of your room, with the temperature (°C) logged every 10 minutes
P_MAX = 8.0              # power of the heater (kW), used at full power in the test
P_INT0 = 0.3             # internal gains of the empty room (kW): lights and devices
T_OUT_TEST = 3.0         # mean outdoor temperature during the test (°C)
STEP_LOG = [11.0, 11.9, 12.7, 13.5, 14.3, 15.0, 15.8, 16.6, 17.0, 17.6, 18.3, 18.8,
            19.2, 19.6, 20.1, 20.6, 20.8, 21.3, 21.5, 21.9, 22.1, 22.6, 22.8, 23.2,
            23.5, 23.7, 23.6, 24.1, 24.3, 24.5, 24.5]

def step_curve(t, T_inf, tau, T0):
    """Temperature after a step of the heater power: T_inf - (T_inf - T0) exp(-t / tau)."""
    return T_inf - (T_inf - T0) * np.exp(-t / tau)

t_test = np.arange(len(STEP_LOG)) * 10 / 60            # hours since the heater was switched on
(T_inf, tau, T0), cov = curve_fit(step_curve, t_test, STEP_LOG, p0=[25.0, 2.0, STEP_LOG[0]])
R = (T_inf - T_OUT_TEST) / (P_MAX + P_INT0)            # thermal resistance (K/kW)
C = tau / R                                            # thermal capacitance (kWh/K)
fit_rmse = np.sqrt(np.mean((step_curve(t_test, T_inf, tau, T0) - np.array(STEP_LOG)) ** 2))
print(f"T_inf = {T_inf:.2f} °C, tau = {tau:.2f} h (standard error {np.sqrt(cov[1, 1]):.2f} h)")
print(f"R = {R:.3f} K/kW, C = {C:.3f} kWh/K, RMSE of the fit = {fit_rmse:.3f} K")
RESULTS["R (K/kW)"] = round(R, 3)
RESULTS["C (kWh/K)"] = round(C, 3)
RESULTS["tau (h)"] = round(tau, 2)

tt = np.linspace(0, t_test[-1], 200)
plt.figure(figsize=(6.5, 3.2))
plt.plot(t_test, STEP_LOG, "o", ms=4, label="logged")
plt.plot(tt, step_curve(tt, T_inf, tau, T0), label="fitted model")
plt.xlabel("Time since the heater was switched on (h)")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The next cell describes a typical winter day of the room: The outdoor temperature, the internal gains with
the occupancy and the setpoints of the heating. The function `room_step` advances the model by one minute with the
Euler method. The cell also checks the size of the heater. At every minute of the day, the power
$(r - T_{out})/R - P_i$ holds the room at its setpoint $r$, and the heater must be able to deliver the largest of these
values.

In [ ]:
# TODO: a typical winter day of your room
DT = 1 / 60                       # simulation step (h): one minute
HOURS = np.arange(0, 24, DT)      # the minutes of one day, in hours
OCCUPIED = (8, 18)                # hours in which the comfort is assessed
COMFORT = (20.0, 23.0)            # comfort band (°C)

def t_out(h):
    """Outdoor temperature (°C) at the hour h: -2 °C at 3:00 and 6 °C at 15:00."""
    return 2.0 + 4.0 * np.sin(2 * np.pi * (h - 9) / 24)

def gains(h):
    """Internal gains (kW): lights and devices, plus 20 people of 0.1 kW each from 9:00 to 17:00."""
    return P_INT0 + (2.0 if 9 <= h < 17 else 0.0)

def setpoint(h):
    """Setpoint (°C): 21 °C from 7:00 to 18:00 and a setback to 16 °C at night."""
    return 21.0 if 7 <= h < 18 else 16.0

def room_step(T, P, h):
    """One Euler step of C dT/dt = (T_out - T) / R + P + P_i."""
    return T + DT * ((t_out(h) - T) / R + P + gains(h)) / C

need = max((setpoint(h) - t_out(h)) / R - gains(h) for h in HOURS)
print(f"Largest power needed to hold the setpoint: {need:.2f} kW (heater: {P_MAX} kW)")
RESULTS["largest power needed (kW)"] = round(need, 2)

## 2. A thermostat with a noisy sensor

*Lecture: [Week 3, Hybrid automata](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#hybrid-automata) and [Week 4, Quantisation and noise](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#quantisation)*

A thermostat is a hybrid automaton with two modes. In the mode *Heat*, the heater runs at full power, and in the mode
*Idle* it is off. The guard $y \le r - h$ switches to *Heat*, and the guard $y \ge r + h$ switches back to *Idle*. Here
$y$ is the sensor reading, $r$ the setpoint and $h$ the half width of the hysteresis band. In each mode, the temperature
follows the model of Section 1.

The sensor samples every $T_s$ minutes. Its reading has noise with the standard deviation $\sigma$ and is rounded to the
resolution $q$, which adds quantisation noise with the standard deviation $q/\sqrt{12}$. A narrow band lets the noise
switch the heater on and off many times, which wears out its contactor. A wide band lets the temperature swing. The cell
sweeps the band and counts the switchings of one day. It then estimates the life of the contactor from its rated number
of operations.

In [ ]:
# TODO: the sensor and the thermostat of your room
TS_MIN = 1                 # sampling period of the sensor (min)
SIGMA = 0.15               # standard deviation of the sensor noise (K)
Q_RES = 0.0625             # resolution of the sensor (K), as in a 12-bit digital sensor
H_BAND = 0.3               # half width of the hysteresis band (K)
CONTACTOR_LIFE = 100_000   # rated switching operations of the contactor of the heater
SEASON_DAYS = 180          # heating days per year

def read_sensor(T, rng):
    """A noisy reading of the temperature T, rounded to the resolution of the sensor."""
    return Q_RES * np.round((T + rng.normal(0, SIGMA)) / Q_RES)

def run_thermostat(h_band, T0=16.0, seed=1):
    """One day of the room with the thermostat; returns the log and the number of switchings."""
    rng = np.random.default_rng(seed)
    T, heat, y, switches, log = T0, False, T0, 0, []
    for k, h in enumerate(HOURS):
        if k % TS_MIN == 0:                          # a new reading every TS_MIN minutes
            y, r = read_sensor(T, rng), setpoint(h)
            if heat and y >= r + h_band:             # guard from Heat to Idle
                heat, switches = False, switches + 1
            elif not heat and y <= r - h_band:       # guard from Idle to Heat
                heat, switches = True, switches + 1
        P = P_MAX if heat else 0.0
        log.append((h, T, y, P))
        T = room_step(T, P, h)
    return pd.DataFrame(log, columns=["hour", "temp", "reading", "power"]), switches

def metrics(df):
    """Energy and comfort of one day."""
    occ = df[(df.hour >= OCCUPIED[0]) & (df.hour < OCCUPIED[1])]
    return {"energy (kWh)": round(df.power.sum() * DT, 1),
            "in comfort (%)": round(100 * occ.temp.between(*COMFORT).mean(), 1),
            "K h below the band": round(np.clip(COMFORT[0] - occ.temp, 0, None).sum() * DT, 2),
            "highest (°C)": round(occ.temp.max(), 2)}

print(f"Standard deviation of a reading: {np.sqrt(SIGMA ** 2 + Q_RES ** 2 / 12):.3f} K")
rows = []
for hb in [0.05, 0.1, 0.2, 0.3, 0.5, 0.8]:
    df, n = run_thermostat(hb)
    rows.append({"band (± K)": hb, "switchings per day": n, **metrics(df)})
print(pd.DataFrame(rows).to_string(index=False))

thermo, n_sw = run_thermostat(H_BAND)
years = CONTACTOR_LIFE / (n_sw * SEASON_DAYS)
print(f"Band ± {H_BAND} K: {n_sw} switchings per day, life of the contactor {years:.1f} years")
RESULTS["thermostat: switchings per day"] = n_sw
RESULTS["thermostat: life of the contactor (years)"] = round(years, 1)
RESULTS["thermostat: in comfort (%)"] = metrics(thermo)["in comfort (%)"]
RESULTS["thermostat: energy (kWh)"] = metrics(thermo)["energy (kWh)"]

## 3. A proportional-integral controller with anti-windup

*Lecture: [Week 5, Integral action](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#pi-control) and [Week 5, Stability](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#stability) and [Week 5, Saturation and windup](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#windup)*

A heater with a power controller, such as a thyristor unit, can deliver any power between 0 and $P_{max}$. A
proportional-integral (PI) controller sets the power from the error $e = r - y$:

$$P = K_p\,e + K_i \int e\,dt .$$

With the model of Section 1, the closed loop has the characteristic equation $C s^2 + (1/R + K_p)\,s + K_i = 0$. Its
poles take a chosen damping ratio $\zeta$ and natural frequency $\omega$ with

$$K_i = C\,\omega^2, \qquad K_p = 2\,\zeta\,\omega\,C - 1/R .$$

The heater cannot deliver less than 0 or more than $P_{max}$. While the output is saturated, the integral keeps growing
and causes an overshoot later, the windup. The anti-windup stops the integration while the output is saturated in the
direction of the error. The cell compares the thermostat of Section 2 with the PI controller, with and without the
anti-windup. The total change of the heater power over the day measures how much noise reaches the actuator.

In [ ]:
# TODO: the design of your PI controller
ZETA = 1.0               # damping ratio of the closed loop
T_CL = 0.4               # time constant of the closed loop, 1 / omega (h)

omega = 1 / T_CL
KI = C * omega ** 2                      # integral gain (kW per K h)
KP = 2 * ZETA * omega * C - 1 / R        # proportional gain (kW/K)
print(f"Kp = {KP:.2f} kW/K, Ki = {KI:.2f} kW/(K h), poles {np.round(np.roots([C, 1 / R + KP, KI]), 3)} 1/h")

def run_pi(kp, ki, anti_windup=True, estimator=None, T0=16.0, seed=1):
    """One day of the room with the PI controller; an estimator, if given, filters the readings."""
    rng = np.random.default_rng(seed)
    T, I, P, x, log = T0, 0.0, 0.0, T0, []
    est = estimator() if estimator else None
    for k, h in enumerate(HOURS):
        if k % TS_MIN == 0:
            y = read_sensor(T, rng)
            x = est(y, P, h) if est else y           # the value that the controller uses
            e = setpoint(h) - x
            u = kp * e + ki * I
            P = min(max(u, 0.0), P_MAX)
            saturated = (u > P_MAX and e > 0) or (u < 0 and e < 0)
            if not (anti_windup and saturated):
                I += e * TS_MIN * DT                 # integral of the error (K h)
        log.append((h, T, x, P))
        T = room_step(T, P, h)
    return pd.DataFrame(log, columns=["hour", "temp", "reading", "power"])

def variation(df):
    """Total change of the heater power over the day (kW)."""
    return round(float(np.abs(np.diff(df.power)).sum()), 1)

pi_aw = run_pi(KP, KI, anti_windup=True)
pi_no = run_pi(KP, KI, anti_windup=False)
table = pd.DataFrame([
    {"controller": f"thermostat, band ± {H_BAND} K", **metrics(thermo), "power variation (kW)": variation(thermo)},
    {"controller": "PI with anti-windup", **metrics(pi_aw), "power variation (kW)": variation(pi_aw)},
    {"controller": "PI without anti-windup", **metrics(pi_no), "power variation (kW)": variation(pi_no)}])
print(table.to_string(index=False))
RESULTS["PI: in comfort (%)"] = metrics(pi_aw)["in comfort (%)"]
RESULTS["PI: energy (kWh)"] = metrics(pi_aw)["energy (kWh)"]
RESULTS["PI: highest (°C)"] = metrics(pi_aw)["highest (°C)"]
RESULTS["PI without anti-windup: highest (°C)"] = metrics(pi_no)["highest (°C)"]
RESULTS["PI: power variation (kW)"] = variation(pi_aw)

fig, ax = plt.subplots(2, 1, figsize=(8, 5.2), sharex=True)
for df, label in [(thermo, "thermostat"), (pi_aw, "PI with anti-windup"), (pi_no, "PI without anti-windup")]:
    ax[0].plot(df.hour, df.temp, label=label)
    ax[1].plot(df.hour, df.power, lw=0.8, label=label)
ax[0].plot(HOURS, [setpoint(h) for h in HOURS], "k--", lw=1, label="setpoint")
ax[0].axhspan(*COMFORT, color="green", alpha=0.08)
ax[0].set_ylabel("Temperature (°C)")
ax[1].set_ylabel("Heater power (kW)")
ax[1].set_xlabel("Hour of the day")
ax[0].legend(fontsize=8, ncol=2)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. A Kalman filter for the readings

*Lecture: [Week 6, The Kalman filter: Predict and correct](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#kalman)*

The PI controller passes the noise of the sensor to the heater. A Kalman filter combines the model of Section 1 with
the readings. Over one sampling period $T_s$ in hours, the model predicts the temperature

$$\hat{x}^- = a\,\hat{x} + (1 - a)\,T_\infty, \qquad a = e^{-T_s/\tau},$$

where $T_\infty = T_{out} + R\,(P + P_i)$ uses the inputs of the period. The variance of the prediction is
$p^- = a^2 p + Q$. The reading $y$ corrects the prediction with the gain $K = p^-/(p^- + V)$, where
$V = \sigma^2 + q^2/12$ is the variance of a reading:

$$\hat{x} = \hat{x}^- + K\,(y - \hat{x}^-), \qquad p = (1 - K)\,p^- .$$

The filter needs the internal gains for its prediction. A model of the empty room misses the people, and with a
small $Q$ the estimate then lags behind the true temperature. With the timetable of the room, the model error is
small, and a small $Q$ smooths the readings strongly. The cell sweeps $Q$ for both models and runs the PI
controller of Section 3 on the filtered value. It compares the RMSE against the true temperature, the power
variation and the highest temperature with the raw readings.

In [ ]:
# TODO: the process noise of your filter and what it knows about the internal gains
Q_PROC = 1e-4            # variance of the model error per sampling period (K²)
KNOWN_GAINS = gains      # the timetable of Section 1; use lambda h: P_INT0 for the empty room
V_SENSOR = SIGMA ** 2 + Q_RES ** 2 / 12

def make_kalman(q=Q_PROC, known=KNOWN_GAINS):
    """A scalar Kalman filter of the room temperature with the model of Section 1."""
    a = np.exp(-TS_MIN * DT / (R * C))
    state = {"x": None, "p": V_SENSOR, "h": None}

    def step(y, P, h):
        if state["x"] is None:                       # the first reading starts the filter
            state["x"] = y
        else:
            h0 = state["h"]
            x_pred = a * state["x"] + (1 - a) * (t_out(h0) + R * (P + known(h0)))
            p_pred = a ** 2 * state["p"] + q
            K = p_pred / (p_pred + V_SENSOR)
            state["x"] = x_pred + K * (y - x_pred)
            state["p"] = (1 - K) * p_pred
        state["h"] = h
        return state["x"]
    return step

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

rows = []
for label, known in [("empty room", lambda h: P_INT0), ("timetable", gains)]:
    for q in [1e-4, 4e-4, 1e-3, 4e-3]:
        df = run_pi(KP, KI, estimator=lambda q=q, known=known: make_kalman(q, known))
        rows.append({"model of the gains": label, "Q (K²)": q, "RMSE (K)": round(rmse(df.reading, df.temp), 3),
                     "power variation (kW)": variation(df), "highest (°C)": metrics(df)["highest (°C)"]})
print(f"Raw readings: RMSE {rmse(pi_aw.reading, pi_aw.temp):.3f} K, power variation {variation(pi_aw)} kW, "
      f"highest {metrics(pi_aw)['highest (°C)']} °C")
print(pd.DataFrame(rows).to_string(index=False))

pi_kf = run_pi(KP, KI, estimator=make_kalman)
RESULTS["raw readings: RMSE (K)"] = round(rmse(pi_aw.reading, pi_aw.temp), 3)
RESULTS["Kalman filter: RMSE (K)"] = round(rmse(pi_kf.reading, pi_kf.temp), 3)
RESULTS["PI on the filter: power variation (kW)"] = variation(pi_kf)
RESULTS["PI on the filter: in comfort (%)"] = metrics(pi_kf)["in comfort (%)"]

fig, ax = plt.subplots(figsize=(8, 3.2))
w = (pi_kf.hour >= 9) & (pi_kf.hour < 11)
ax.plot(pi_aw.hour[w], pi_aw.reading[w], lw=0.8, color="#9AA5AE", label="raw readings")
ax.plot(pi_kf.hour[w], pi_kf.reading[w], color="#1E6FA8", label="Kalman filter")
ax.plot(pi_kf.hour[w], pi_kf.temp[w], "k--", lw=1, label="true temperature")
ax.set_xlabel("Hour of the day")
ax.set_ylabel("Temperature (°C)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. The timing of the controller

*Lecture: [Week 7, Tasks on one processor](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#tasks) and [Week 7, Rate-monotonic scheduling](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#rate-monotonic)*

The microcontroller of the heating runs several periodic tasks. Each task $i$ has a worst-case execution time $C_i$ and
a period $T_i$, and its deadline is the end of its period. Rate-monotonic scheduling gives the highest priority to the
shortest period. A utilisation $U = \sum_i C_i / T_i$ below the bound $n\,(2^{1/n} - 1)$ proves that all $n$ tasks meet
their deadlines. Above the bound, the response-time analysis decides. The worst-case response time of task $i$ is the
smallest solution of

$$R_i = C_i + \sum_{j \in hp(i)} \left\lceil \frac{R_i}{T_j} \right\rceil C_j ,$$

where $hp(i)$ are the tasks of higher priority. Measure the execution times on your board, for example with a timer
around each task, and add a margin. Without a board, take values from the documentation of a similar controller and
say so in the report.

In [ ]:
# TODO: the periodic tasks of your controller, with execution times measured on the board (ms)
TASKS = pd.DataFrame([
    ("Read and average the sensor",     4.0,     50),
    ("Update the display",             60.0,    200),
    ("Log to the memory card",         90.0,   1000),
    ("Report over Wi-Fi",            1500.0,   5000),
    ("Kalman filter and PI step",      12.0,  60000),
], columns=["task", "C (ms)", "T (ms)"])

def response_times(tasks):
    """Worst-case response times under rate-monotonic priorities (shortest period first)."""
    tasks = tasks.sort_values("T (ms)", kind="stable").reset_index(drop=True)
    out = []
    for i, row in tasks.iterrows():
        hp = tasks.iloc[:i]
        r = row["C (ms)"]
        while True:
            r_new = row["C (ms)"] + sum(math.ceil(r / t) * c for c, t in zip(hp["C (ms)"], hp["T (ms)"]))
            if r_new == r or r_new > row["T (ms)"]:
                break
            r = r_new
        out.append(r_new)
    tasks["R (ms)"] = out
    tasks["deadline met"] = tasks["R (ms)"] <= tasks["T (ms)"]
    return tasks

n = len(TASKS)
U = float((TASKS["C (ms)"] / TASKS["T (ms)"]).sum())
bound = n * (2 ** (1 / n) - 1)
rt = response_times(TASKS)
print(f"Utilisation U = {U:.3f}, bound for {n} tasks = {bound:.3f}")
print(rt.to_string(index=False))
RESULTS["utilisation"] = round(U, 3)
RESULTS["bound of the utilisation"] = round(bound, 3)
RESULTS["all deadlines met"] = bool(rt["deadline met"].all())
RESULTS["response time of the control task (ms)"] = float(rt.loc[rt.task == "Kalman filter and PI step", "R (ms)"].iloc[0])

Under rate-monotonic priorities, the control step runs last, because its period is the longest. Its response time is still far below its period of one minute. Compare it with the time constant of the room, and discuss whether a fixed priority for the control step would be better.

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>